In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os

def visualize_voxel_from_pt(pt_path, save_path=None):
    # 加载 .pt 文件
    data = torch.load(pt_path, map_location='cpu')

    print(f"Loaded data from {pt_path}")
    print(data.keys())
    for keys in data.keys():
        if isinstance(data[keys], torch.Tensor):
            print(f"  {keys}: {data[keys].shape} (Tensor)")
        else:
            print(f"  {keys}: {data[keys]}")        
    
    # 体素网格应为 key: 'Volume'，形状 [32, 32, 32]，或 [1, 32, 32, 32]
    volume = data['Volume']
    if isinstance(volume, torch.Tensor):
        volume = volume.squeeze().numpy()
    else:
        volume = np.array(volume)

    # 获取非零体素的位置
    xs, ys, zs = np.nonzero(volume > 0)

    # 创建图形
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(xs, ys, zs, marker='s', alpha=0.6, s=20, c='black')

    ax.set_title(f"Voxel Visualization of {os.path.basename(pt_path)}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_xlim(0, 32)
    ax.set_ylim(0, 32)
    ax.set_zlim(0, 32)
    plt.tight_layout()

    # 如果指定了保存路径，则保存图像
    if save_path:
        print(f"Saved figure to: {save_path}")

    plt.show()

# 示例调用：指定 .pt 文件路径和保存路径
visualize_voxel_from_pt(
    'SNC_valid/train/airplane_02691156_0007.pt',
    save_path='voxel_visualization.png'
)



In [ ]:
import os
import shutil
from pathlib import Path

def organize_mat_files(source_dir, output_root='SNC'):
    # 遍历所有 .mat 文件
    for mat_file in Path(source_dir).rglob("*.mat"):
        filename = mat_file.name  # 例如 train0_pc_00000.mat
        prefix = filename.split('_pc_')[0]  # 提取如 train0 / test2

        # 决定属于 train/test/val
        if prefix.startswith("train"):
            split_type = "train"
        elif prefix.startswith("test"):
            split_type = "test"
        elif prefix.startswith("val"):
            split_type = "val"
        else:
            print(f"跳过未识别 split 类型的文件: {filename}")
            continue

        # 构造目标路径
        target_folder = Path(output_root) / split_type / prefix
        target_folder.mkdir(parents=True, exist_ok=True)

        # 拷贝文件到新目录
        target_path = target_folder / filename
        shutil.move(str(mat_file), str(target_path))

        print(f"移动: {filename} → {target_path}")

organize_mat_files('SNC_valid', output_root='SNC_valid')


In [ ]:
import os
import shutil
from pathlib import Path

def organize_mat_files_flat(source_dir, output_root='SNC_pt'):
    # 遍历所有 .mat 文件
    for mat_file in Path(source_dir).rglob("*.pt"):
        filename = mat_file.name  # 例如 train0_pc_00000.mat
        prefix = filename.split('_pc_')[0]  # 提取如 train0、test1 等

        # 识别属于哪一类
        if prefix.startswith("train"):
            split_type = "train"
        elif prefix.startswith("test"):
            split_type = "test"
        elif prefix.startswith("val"):
            split_type = "val"
        else:
            print(f"跳过未识别 split 类型的文件: {filename}")
            continue

        # 构造目标路径（无子文件夹）
        target_folder = Path(output_root) / split_type
        target_folder.mkdir(parents=True, exist_ok=True)

        # 移动文件
        target_path = target_folder / filename
        shutil.move(str(mat_file), str(target_path))

        print(f"移动: {filename} → {target_path}")

# 使用方式
organize_mat_files_flat('SNC_pt', output_root='SNC_pt')


In [ ]:
import os
import glob
from collections import Counter

# 设置文件夹路径
train_dir = "./SNC_valid/train"
test_dir = "./SNC_valid/test"

# 匹配所有 pt 文件（train + test）
file_paths = glob.glob(os.path.join(train_dir, "*.pt")) + \
             glob.glob(os.path.join(test_dir, "*.pt"))

# 统计 name 的出现次数
name_counter = Counter()

for path in file_paths:
    filename = os.path.basename(path)  # 只拿文件名部分
    parts = filename.split("_")
    if len(parts) >= 3:
        name = parts[0]  # 取 name
        name_counter[name] += 1
    else:
        print(f"⚠️ 文件名格式异常，跳过：{filename}")

# 打印统计结果
print("📊 类别统计（按 name 分组）：")
for name, count in name_counter.items():
    print(f"{name}: {count}")
